# Data Wrangling com pandas (MBA USP/Esalq)
Resumo das funções de *data wrangling* da disciplina (renomear, selecionar, criar e categorizar variáveis, estatísticas, filtros, *merge*, duplicados, *melt* e encadeamento) e as atividades práticas com os indicadores do Banco Mundial e os catálogos de filmes e séries de *streaming*.

**Origem:** MBA em Data Science e Analytics USP/Esalq, disciplina de Data Wrangling (Prof. Dr. Wilson Tarantin Junior), complementado com os *datasets* do Banco Mundial publicados no Kaggle.

**Dados:**
- Enviar para `/content/` no Colab (arquivos da disciplina): `(1.2) dataset_principal.xls`, `(1.3) dataset_join.xls`, `(2.2) WDI World Bank.xlsx`, `(2.3) WDI Income Group.xlsx`, `(2.4) WDI Country.xlsx`, `(5.2) Filmes Streaming.csv` e `(5.3) Séries Streaming.csv` (estes dois vêm do [Kaggle](https://www.kaggle.com/datasets/ruchi798/tv-shows-on-netflix-prime-video-hulu-and-disney)).
- Baixados pelo próprio notebook: [World Bank datasets](https://www.kaggle.com/datasets/nilaychauhan/world-bank-datasets) (via `kagglehub`) e a API do Banco Mundial.

**Índice**
1. Funções básicas de *data wrangling*
2. World Bank: indicadores de saúde (WDI) e *datasets* do Kaggle
3. *Streaming*: filmes e séries

In [ ]:
# Setup: pycountry não vem instalado no Colab
!pip install -q pycountry

import os
import sqlite3
from collections import defaultdict

import kagglehub
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pycountry import countries

# Opcional (somente no Colab): tabelas interativas para DataFrames
# from google.colab import data_table
# data_table.enable_dataframe_formatter()

pd.set_option('display.max.columns', None)  # mostra todas as colunas

## 1. Funções básicas de *data wrangling*
Base da disciplina: tempo que estudantes levam para chegar à escola, com distância, quantidade de semáforos, período do dia e perfil ao volante.

### 1.1 Carga e visão geral

In [ ]:
dados_tempo = pd.read_excel('/content/(1.2) dataset_principal.xls')
dados_merge = pd.read_excel('/content/(1.3) dataset_join.xls')

lista_colunas = pd.Series(dados_tempo.columns)  # nomes das colunas como Series
print(lista_colunas)
dados_tempo.info()

In [ ]:
display(dados_tempo.head(5))  # primeiras linhas
dados_tempo.tail(5)           # últimas linhas

### 1.2 Renomeando colunas
`rename` recebe um dicionário `{nome_antigo: nome_novo}`; também é possível referenciar a coluna pela posição (`df.columns[i]`). Com `inplace=True` a alteração é feita no próprio objeto e o método retorna `None`, por isso não se atribui o resultado a uma variável.

In [ ]:
dados_tempo = dados_tempo.rename(columns={'Estudante': 'estudante',
                                          'Tempo para chegar à escola (minutos)': 'tempo',
                                          'Distância percorrida até a escola (quilômetros)': 'distancia',
                                          'Quantidade de semáforos': 'semaforos',
                                          'Período do dia': 'periodo',
                                          'Perfil ao volante': 'perfil'})

# Renomeando pela posição das colunas, em um novo DataFrame
dados_novo = dados_tempo.rename(columns={dados_tempo.columns[0]: 'obs',
                                         dados_tempo.columns[1]: 'temp',
                                         dados_tempo.columns[5]: 'perf'})
dados_novo.head()

### 1.3 Selecionando linhas e colunas
`iloc[linhas, colunas]` seleciona por posição (o fim do intervalo não é incluído); colchetes simples selecionam uma coluna (`Series`) e duplos uma lista de colunas (`DataFrame`).

In [ ]:
display(dados_tempo.iloc[3, ])       # quarta linha, todas as colunas
display(dados_tempo.iloc[:, 4])      # quinta coluna, todas as linhas
display(dados_tempo.iloc[2:5, ])     # linhas nas posições 2 a 4
display(dados_tempo.iloc[:, 3:5])    # colunas nas posições 3 e 4
display(dados_tempo.iloc[2:4, 3:5])  # linhas 2 e 3, colunas 3 e 4
dados_tempo.iloc[5, 4]               # um único valor

In [ ]:
var_tempo = dados_tempo['tempo']                    # equivalente a dados_tempo.tempo
var_tempo_perfil = dados_tempo[['tempo', 'perfil']]

# Selecionando variáveis por um início ou fim em comum
selec_1 = dados_tempo.loc[:, dados_tempo.columns.str.startswith('per')]
selec_2 = dados_tempo.loc[:, dados_tempo.columns.str.endswith('fil')]
selec_1.head()

### 1.4 Criando colunas e linhas
Novas colunas podem vir de uma `Series` (alinhada pelo índice) ou de operações entre colunas; novas linhas são adicionadas com `pd.concat`. Para as linhas, o DataFrame novo precisa ter os mesmos nomes de colunas.

In [ ]:
dados_novo = dados_tempo.rename(columns={'estudante': 'obs',
                                         'tempo': 'temp',
                                         'distancia': 'dist',
                                         'semaforos': 'sem',
                                         'periodo': 'per',
                                         'perfil': 'perf'})

idade = pd.Series([25, 28, 30, 19, 20, 36, 33, 48, 19, 21])
dados_novo['idade'] = idade
dados_novo['sem_km'] = round(dados_novo['sem'] / dados_novo['dist'], 2)

nova_obs = pd.DataFrame({'per': ['Tarde'],
                         'obs': ['Roberto'],
                         'temp': [40]})
dados_concat = pd.concat([dados_novo, nova_obs], ignore_index=True)
dados_concat.tail()

### 1.5 Recodificando e categorizando variáveis
`map` troca valores segundo um dicionário; `np.where` aninhado cria categorias por condições; `pd.qcut` divide a variável em quantis.

In [ ]:
labels = {'calmo': 'perfil_A',
          'moderado': 'perfil_B',
          'agressivo': 'perfil_C'}

numeros = {'calmo': 1,
           'moderado': 2,
           'agressivo': 3}

df_labels = dados_tempo.assign(novo_perfil=dados_tempo['perfil'].map(labels))
df_numeros = dados_tempo.assign(novo_perfil=dados_tempo['perfil'].map(numeros))

# Variável numérica transformada em categórica e exclusão de colunas
df_numeros['novo_perfil'] = df_numeros['novo_perfil'].astype('category')
df_numeros = df_numeros.drop(columns=['periodo', 'perfil'])
df_numeros.info()

In [ ]:
dados_tempo['faixa'] = np.where(dados_tempo['tempo'] <= 20, 'rápido',
                       np.where((dados_tempo['tempo'] > 20) & (dados_tempo['tempo'] <= 40), 'médio',
                       np.where(dados_tempo['tempo'] > 40, 'demorado',
                                'demais')))

# Categorização pelos quartis da variável (q=4)
dados_tempo['quartis'] = pd.qcut(dados_tempo['tempo'], q=4, labels=['q1', 'q2', 'q3', 'q4'])
dados_tempo.head()

### 1.6 Ordenação

In [ ]:
df_org_1 = dados_tempo.sort_values(by=['tempo', 'perfil'], ascending=[True, False]).reset_index(drop=True)
df_org_1.head()

### 1.7 Estatísticas descritivas, frequências e agrupamentos

In [ ]:
display(dados_tempo.describe())  # .T para transpor a tabela
dados_tempo['tempo'].agg(['count', 'mean', 'median', 'std', 'min', 'max', 'sum'])

In [ ]:
display(dados_tempo['tempo'].quantile([0.25, 0.75]))
dados_tempo[['tempo', 'distancia', 'semaforos']].corr()  # correlação de Pearson

In [ ]:
display(dados_tempo['periodo'].value_counts())                # frequência absoluta
display(dados_tempo['perfil'].value_counts(normalize=True))   # frequência relativa
print(dados_tempo['tempo'].unique(), dados_tempo['tempo'].nunique(), dados_tempo.size)

# Tabela de frequências cruzadas para pares de variáveis qualitativas
pd.crosstab(dados_tempo['periodo'], dados_tempo['perfil'])  # normalize=True para proporções

In [ ]:
display(dados_tempo.groupby('periodo')['tempo'].mean())

# Agregando estatísticas diferentes por coluna
dados_tempo.agg({'tempo': 'mean',
                 'distancia': 'mean',
                 'periodo': 'count'})

### 1.8 Filtros
Filtros podem ser feitos com máscaras booleanas ou com `query`. Operadores: `==` (igual), `!=` (diferente), `&` (e), `|` (ou), `>`, `<`, `>=`, `<=`. Em `query`, variáveis do ambiente são referenciadas com `@`.

In [ ]:
filtro_calmo = dados_tempo[dados_tempo['perfil'] == 'calmo']
filtro_calmo_q1 = dados_tempo[(dados_tempo['perfil'] == 'calmo') & (dados_tempo['quartis'] == 'q1')]

# Mesmo filtro com query
display(dados_tempo.query('perfil == "calmo" & quartis == "q1"'))
display(dados_tempo.query('tempo >= 30'))
dados_tempo[dados_tempo['tempo'].between(25, 30, inclusive='both')]

In [ ]:
nomes = pd.Series(['Gabriela', 'Leonor'])
display(dados_tempo[dados_tempo['estudante'].isin(nomes)])
display(dados_tempo.query('estudante.isin(@nomes)'))
dados_tempo.query('not perfil.isin(["calmo", "moderado"])')

### 1.9 Junção de bases (*merge*)
`how='left'` mantém todas as linhas da base da esquerda; também há `right`, `inner` e `outer`. A chave precisa ter o mesmo nome nas duas bases.

In [ ]:
dados_merge = dados_merge.rename(columns={'Estudante': 'estudante'})
merge = pd.merge(dados_tempo, dados_merge, on='estudante', how='left')
merge.head()

### 1.10 Duplicados e valores faltantes

In [ ]:
print('Linhas duplicadas:', len(dados_tempo) - len(dados_tempo.drop_duplicates()))
sem_duplicadas = dados_tempo.drop_duplicates(subset=['estudante', 'tempo'])

print(merge.isna().sum())
merge.fillna('texto genérico').head()

### 1.11 *Melt*: colocando colunas uma abaixo da outra

In [ ]:
dados_melt = pd.melt(dados_tempo, id_vars=['estudante'], value_vars=['tempo', 'perfil'])
dados_melt.head()

### 1.12 Encadeamento de funções
Os métodos podem ser encadeados entre parênteses; dentro de `assign`, uma `lambda` recebe o DataFrame do passo anterior.

In [ ]:
resumo = (dados_tempo
          .assign(sem_km=lambda d: round(d['semaforos'] / d['distancia'], 2))
          .query('tempo >= 30')
          .rename(columns={'periodo': 'per'})
          .groupby('per')
          .agg({'sem_km': 'mean',
                'distancia': 'mean',
                'estudante': 'count'}))
resumo

## 2. World Bank: indicadores de saúde (WDI) e *datasets* do Kaggle
### 2.1 WDI: indicadores de saúde por país (atividade da disciplina)
A planilha WDI tem uma linha por país × série (indicador) com o valor de 2021. Os dados são filtrados para o tópico *Health*, pivotados (cada série vira coluna) e cruzados com as informações dos países e do grupo de renda. Na leitura, `na_values='..'` trata `..` como valor ausente.

In [ ]:
dados_wdi = pd.read_excel('/content/(2.2) WDI World Bank.xlsx', na_values='..')
dados_grupo = pd.read_excel('/content/(2.3) WDI Income Group.xlsx')
dados_paises = pd.read_excel('/content/(2.4) WDI Country.xlsx')

dados_wdi.rename(columns={'Country Name': 'pais',
                          'Country Code': 'cod_pais',
                          'Series Name': 'serie',
                          'Series Code': 'cod_serie',
                          '2021 [YR2021]': 'ano_2021',
                          'Topic': 'topico'}, inplace=True)
dados_paises.columns

In [ ]:
# Mantendo apenas os dados até a linha 383.571 (as últimas linhas são notas de rodapé)
dados_wdi = dados_wdi.iloc[0:383572, ]
dados_wdi['pais'].tail(n=20)

In [ ]:
# Séries do tópico Health, pivotadas: cada série vira uma coluna, uma linha por país
dados_saude = dados_wdi[dados_wdi['topico'].str.startswith('Health')]
dados_saude = pd.pivot(dados_saude,
                       index=['pais', 'cod_pais'],
                       columns=['serie'],
                       values='ano_2021')

# Junção com as informações dos países
dados_paises.rename(columns={'Country': 'cod_pais'}, inplace=True)
dados_saude = pd.merge(dados_saude, dados_paises, on='cod_pais', how='left')

# Mantém só os países (linhas com Name preenchido) e remove a coluna Name
dados_saude = dados_saude[~dados_saude['Name'].isna()].reset_index(drop=True)
dados_saude.drop(columns=['Name'], inplace=True)

# Remove as colunas com N/A em todas as linhas (axis=1 é coluna e axis=0 é linha)
dados_saude.dropna(axis=1, how='all', inplace=True)

In [ ]:
# Grupo de renda: seleciona as variáveis, renomeia a chave e junta
dados_grupo_select = dados_grupo[['Code', 'Income Group']].copy()
dados_grupo_select.rename(columns={'Code': 'cod_pais'}, inplace=True)
dados_saude = pd.merge(dados_saude, dados_grupo_select, how='left', on='cod_pais')

# Reposiciona a variável de grupo na terceira coluna
organizar = dados_saude.pop('Income Group')
dados_saude.insert(2, 'Group', organizar)

# A variável de interesse está na posição 23
col_pos = dados_saude.columns
print(col_pos[23])
display(dados_saude.iloc[:, 23].describe())

# Estatísticas por grupo de renda
estat_grupo = dados_saude.iloc[:, [2, 23]].groupby('Group').mean().reset_index()
estat_grupo

### 2.2 *Datasets* do Banco Mundial no Kaggle: formatos de importação
O `kagglehub` baixa o *dataset* para uma pasta local e devolve o caminho. Os mesmos dados de população aparecem em CSV, JSON, XML e banco SQLite, e também podem ser consultados pela API do Banco Mundial.

In [ ]:
path = kagglehub.dataset_download('nilaychauhan/world-bank-datasets')
print('Arquivos em', path)
os.listdir(path)

In [ ]:
# CSV (os arquivos de indicadores têm 2 linhas de cabeçalho extra)
df_projects = pd.read_csv(f'{path}/projects_data.csv', dtype=str)
df_electricity = pd.read_csv(f'{path}/electricity_access_percent.csv', header=2)
df_rural = pd.read_csv(f'{path}/rural_population_percent.csv', header=2)
df_indicator = pd.read_csv(f'{path}/population_data.csv', header=2)

In [ ]:
# JSON: verificando a disposição dos dados no arquivo antes de ler
def print_lines(n, file_name):
    with open(file_name) as f:
        for _ in range(n):
            print(f.readline())


print_lines(1, f'{path}/population_data.json')

df_population = pd.read_json(f'{path}/population_data.json', orient='records')
df_population.head()

In [ ]:
# XML: navegando nos registros com BeautifulSoup (mostra os 5 primeiros)
with open(f'{path}/population_data.xml') as fp:
    soup = BeautifulSoup(fp, 'lxml')

for i, record in enumerate(soup.find_all('record')):
    if i == 5:
        break
    for field in record.find_all('field'):
        print(field['name'], ': ', field.text)
    print()

In [ ]:
# Banco SQLite
conn = sqlite3.connect(f'{path}/population_data.db')
df_population_sql = pd.read_sql('SELECT * FROM population_data', conn)
conn.close()
df_population_sql.head()

**API do Banco Mundial:** `https://api.worldbank.org/v2/countries/` + siglas dos países separadas por `;` + `/indicators/` + código do indicador + `?` + opções:
- `per_page`: registros por página;
- `page`: qual página retornar (ex.: 5000 registros com 100 por página);
- `date`: filtro de datas;
- `format`: `json` ou `xml`.

Em JSON, a resposta é uma lista: o primeiro elemento traz a paginação e o segundo, os registros.

In [ ]:
url = 'https://api.worldbank.org/v2/countries/br;cn;us;de/indicators/SP.POP.TOTL/?format=json&per_page=1000'
resposta = requests.get(url)
pd.DataFrame(resposta.json()[1]).head()

### 2.3 Limpeza e transformação
Remoção de colunas vazias (`Unnamed`, geradas pela vírgula no fim de cada linha), união dos indicadores e padronização dos países dos projetos com o código ISO-3.

In [ ]:
df_rural.drop(columns=['Unnamed: 62'], inplace=True)
df_electricity.drop(columns=['Unnamed: 62'], inplace=True)
df_indicator.drop(columns=['Unnamed: 62'], inplace=True)
df_projects.drop(columns=['Unnamed: 56'], inplace=True)

df_population[df_population.isnull().any(axis=1)]

In [ ]:
# Concatenando os dois indicadores em uma única base
df_indicadores = pd.concat([df_rural, df_electricity])
pd.unique(df_indicadores['Indicator Name'])

In [ ]:
# countryname traz nomes repetidos separados por ';' -> mantém só o primeiro
df_projects['Official Country Name'] = df_projects['countryname'].str.split(';').str.get(0)
df_projects['Official Country Name'].drop_duplicates().head()

O `pycountry` procura um país pelo nome (`countries.get(name=...)`) ou por qualquer nome/código conhecido (`countries.lookup(...)`) e devolve, entre outros, o código ISO-3 (`alpha_3`). Os nomes não encontrados são guardados para tratamento manual.

In [ ]:
print(countries.get(name='Spain'))
print(countries.lookup('Kingdom of Spain'))

In [ ]:
country_not_found = []    # nomes não encontrados no pycountry
ISO3 = defaultdict(str)   # dicionário nome -> código ISO-3 (vazio se não houver)

for country in df_projects['Official Country Name'].drop_duplicates().sort_values():
    try:
        ISO3[country] = countries.lookup(country).alpha_3
    except LookupError:
        print(country, ' not found')
        country_not_found.append(country)

In [ ]:
# Quais dos não encontrados existem na base de indicadores (que já tem o código)?
indicator_countries = df_indicator[['Country Name', 'Country Code']].drop_duplicates().sort_values(by='Country Name')

for country in country_not_found:
    if country in indicator_countries['Country Name'].tolist():
        print(country)

In [ ]:
# Complementa o dicionário com os códigos que faltavam
country_not_found_mapping = {
    'South Asia': 'SAS',
    'St. Kitts and Nevis': 'KNA',
    'St. Lucia': 'LCA',
    'St. Vincent and the Grenadines': 'VCT',
    'West Bank and Gaza': 'PSE',
    'World': 'WLD'
}
ISO3.update(country_not_found_mapping)

df_projects['Country Code'] = df_projects['Official Country Name'].apply(lambda x: ISO3[x])
df_projects[df_projects['Country Code'] == '']

In [ ]:
# totalamt: remove as vírgulas de milhar antes de converter para número
df_projects['totalamt'] = df_projects['totalamt'].str.replace(',', '', regex=False)
df_projects['totalamt'] = pd.to_numeric(df_projects['totalamt'], downcast='float')
df_projects['totalamt'].info()

## 3. *Streaming*: filmes e séries
Atividade nº 5 da disciplina: os *datasets* contêm as avaliações (IMDb e Rotten Tomatoes) de filmes e séries disponíveis na Netflix, Prime Video, Hulu e Disney+.

### 3.1 Carga e junção das bases
As duas bases têm estrutura semelhante, mas a de filmes tem colunas a mais; ficam só as 12 primeiras antes de juntá-las.

In [ ]:
dados_filmes = pd.read_csv('/content/(5.2) Filmes Streaming.csv', sep=',')
dados_series = pd.read_csv('/content/(5.3) Séries Streaming.csv', sep=',')

dados_filmes.info()
dados_series.info()

In [ ]:
dados_filmes = dados_filmes.iloc[:, 0:12]
dados_completo = pd.concat([dados_filmes, dados_series], ignore_index=True)
dados_completo.drop(columns=['Unnamed: 0'], inplace=True)
dados_completo

### 3.2 Extraindo as notas do texto
As notas vêm como texto (`7.8/10`, `98/100`): pega-se o início da *string* com `str.slice`, remove-se a barra final com `str.rstrip('/')` e converte-se para `float`.

In [ ]:
dados_completo['Ajuste_IMDB'] = dados_completo['IMDb'].str.slice(0, 4)
dados_completo['Ajuste_Rotten'] = dados_completo['Rotten Tomatoes'].str.slice(0, 3)

dados_completo['Ajuste_IMDB'] = dados_completo['Ajuste_IMDB'].str.rstrip('/').astype('float')
dados_completo['Ajuste_Rotten'] = dados_completo['Ajuste_Rotten'].str.rstrip('/').astype('float')
dados_completo.head()

### 3.3 Estatísticas das notas por tipo

In [ ]:
muda_numeros = {0: 'filme', 1: 'série'}
dados_completo = dados_completo.assign(tipo=dados_completo.Type.map(muda_numeros))

descritivas = dados_completo.groupby(['tipo'])
display(descritivas['Ajuste_IMDB'].describe().T)
descritivas['Ajuste_Rotten'].describe().T

### 3.4 Os "melhores" filmes e séries
São considerados melhores os títulos acima do percentil 95 **nas duas** avaliações. `pd.qcut` com `q=[0, 0.95, 1.0]` divide as notas em "menores" e "maiores" que o percentil 95. A mesma regra é aplicada às séries e aos filmes pela função `melhores_titulos`.

In [ ]:
def melhores_titulos(dados, ordem):
    print(dados[['Ajuste_IMDB', 'Ajuste_Rotten']].quantile(0.95))
    dados = dados.assign(
        Categ_IMDB=pd.qcut(dados.Ajuste_IMDB, q=[0, 0.95, 1.0], labels=['menores', 'maiores']),
        Categ_Rotten=pd.qcut(dados.Ajuste_Rotten, q=[0, 0.95, 1.0], labels=['menores', 'maiores'])
    )
    return dados[(dados['Categ_IMDB'] == 'maiores') &
                 (dados['Categ_Rotten'] == 'maiores')].sort_values(ordem, ascending=False)


melhores_series = melhores_titulos(dados_completo[dados_completo['tipo'] == 'série'], ['Ajuste_Rotten', 'Ajuste_IMDB'])
melhores_series

In [ ]:
melhores_filmes = melhores_titulos(dados_completo[dados_completo['tipo'] == 'filme'], ['Ajuste_IMDB', 'Ajuste_Rotten'])
melhores_filmes